
# **Loan Credit Risk Engine**

This project is built around the Kaggle Home Credit Default Risk Competition.

Many people struggle to get loans due to insufficient or non-existent credit histories. Home Credit strives to broaden financial inclusion for the unbanked population by providing a safe and positive borrowing experience. The objective of this project is to build a machine learning classification model to predict an applicant's repayment capability, ensuring that clients capable of repayment are successfully identified and granted loans.

## **Exploratory Data Analysis**

### 1. Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold


### 2. Loading Train and Test dataset


In [3]:
#Load main table
train = pd.read_csv('../data/application_train.csv')
test = pd.read_csv('../data/application_test.csv')



### 3. Overview of Data


In [4]:
#Target Distribution
print('*** Target Distribution ***')
print(train['TARGET'].value_counts(normalize=True))

#Null Percentages
print('\n*** Null Percentages ***')
null_pct = (train.isnull().sum() / len(train) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0].head(30))

#Dtypes Overview
print('\n*** Dtypes Overview ***')
print(train.dtypes.value_counts())

#Test and Train Shape
print('\n*** Test and Train Shape ***')
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

*** Target Distribution ***
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

*** Null Percentages ***
COMMONAREA_MEDI             69.872297
COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
NONLIVINGAPARTMENTS_MEDI    69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_MODE       68.354953
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MEDI       68.354953
FLOORSMIN_AVG               67.848630
FLOORSMIN_MODE              67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_MEDI            66.497784
YEARS_BUILD_MODE            66.497784
YEARS_BUILD_AVG             66.497784
OWN_CAR_AGE                 65.990810
LANDAREA_MEDI               59.376738
LANDAREA_MODE               59.376738
LANDAREA_AVG                59.376738
BASEMENTAREA_MEDI           58.515956
BASEMENTAREA_AVG            58.515956
BASEMENTAREA_MODE           58.515956
EXT_

## Preprocessing and Anomaly Fix

In [5]:
def process_dataframe(df):
    # 1. Create new features (passing index prevents data scrambling)
    new_features = {
        'DAYS_EMPLOYED_ANOMALY': (df['DAYS_EMPLOYED'] == 365243).astype(int),
        'CREDIT_TO_INCOME_RATIO': df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL'],
        'ANNUITY_TO_INCOME_RATIO': df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL'],
        'CREDIT_TO_ANNUITY_RATIO': df['AMT_CREDIT'] / df['AMT_ANNUITY'],
        'DAYS_EMPLOYED_PERCENT': df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    }

    new_features_df = pd.DataFrame(new_features, index=df.index)

    # 2. Replace the anomaly with NaN
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace({365243: np.nan})

    # 3. Join them to the main dataframe all at once
    return pd.concat([df, new_features_df], axis=1)


print("Engineering features...")
train = process_dataframe(train)
test = process_dataframe(test)


Engineering features...


In [6]:
columns_to_drop = [
    # Highly correlated / Redundant
    'OBS_60_CNT_SOCIAL_CIRCLE',
    'DEF_60_CNT_SOCIAL_CIRCLE',
    'REGION_RATING_CLIENT_W_CITY',
    'AMT_GOODS_PRICE',

    # Near-zero variance (Document flags)
    'FLAG_MOBIL',
    'FLAG_DOCUMENT_2',
    'FLAG_DOCUMENT_4',
    'FLAG_DOCUMENT_5',
    'FLAG_DOCUMENT_6',
    'FLAG_DOCUMENT_7',
    'FLAG_DOCUMENT_8',
    'FLAG_DOCUMENT_9',
    'FLAG_DOCUMENT_10',
    'FLAG_DOCUMENT_11',
    'FLAG_DOCUMENT_12',
    'FLAG_DOCUMENT_13',
    'FLAG_DOCUMENT_14',
    'FLAG_DOCUMENT_15',
    'FLAG_DOCUMENT_16',
    'FLAG_DOCUMENT_17',
    'FLAG_DOCUMENT_18',
    'FLAG_DOCUMENT_19',
    'FLAG_DOCUMENT_20',
    'FLAG_DOCUMENT_21',

    # Redundant Housing Metrics (_MODE and _MEDI)
    'APARTMENTS_MODE', 'APARTMENTS_MEDI',
    'BASEMENTAREA_MODE', 'BASEMENTAREA_MEDI',
    'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BEGINEXPLUATATION_MEDI',
    'YEARS_BUILD_MODE', 'YEARS_BUILD_MEDI',
    'COMMONAREA_MODE', 'COMMONAREA_MEDI',
    'ELEVATORS_MODE', 'ELEVATORS_MEDI',
    'ENTRANCES_MODE', 'ENTRANCES_MEDI',
    'FLOORSMAX_MODE', 'FLOORSMAX_MEDI',
    'FLOORSMIN_MODE', 'FLOORSMIN_MEDI',
    'LANDAREA_MODE', 'LANDAREA_MEDI',
    'LIVINGAPARTMENTS_MODE', 'LIVINGAPARTMENTS_MEDI',
    'LIVINGAREA_MODE', 'LIVINGAREA_MEDI',
    'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MEDI',
    'NONLIVINGAREA_MODE', 'NONLIVINGAREA_MEDI'
]

train.drop(columns=columns_to_drop, inplace=True, errors='ignore')
test.drop(columns=columns_to_drop, inplace=True, errors='ignore')


In [7]:
print("Encoding categorical variables...")

le = LabelEncoder()
le_count = 0

for col in train.columns:
    if train[col].dtype == 'object':
        # If 2 or fewer unique categories
        if len(list(train[col].unique())) <= 2:
            # Train on the training data
            le.fit(train[col])
            # Transform both training and testing data
            train[col] = le.transform(train[col])
            test[col] = le.transform(test[col])

            le_count += 1

print(f'{le_count} columns were label encoded.')


Encoding categorical variables...
0 columns were label encoded.


# LightGBM with Stratified K-Fold

In [8]:
TARGET = 'TARGET'
features = [col for c in train.columns if c not in [TARGET, 'SK_ID_CURR']]

X = train[features]
y = train[TARGET]
X_test = test[features]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

lgb_params = {
    'objective': 'binary',        # Binary classification (default vs. no default)
    'metric': 'auc',              # Competition evaluation metric
    'boosting_type': 'gbdt',

    # Core Learning Params
    'learning_rate': 0.02,        # Slow learning rate for better generalization
    'n_estimators': 5000,         # High tree count (rely on early stopping ~100-200 rounds)

    # Tree Structure (Tuned to prevent overfitting)
    'num_leaves': 34,             # Kept relatively small (2^max_depth is theoretical max)
    'max_depth': 8,               # Restricts tree depth
    'min_child_samples': 40,      # Minimum number of data points in a leaf

    # Sampling / Regularization
    'colsample_bytree': 0.7,      # Uses 70% of features per tree
    'subsample': 0.8,             # Uses 80% of data rows per tree
    'subsample_freq': 1,          # Performs row subsampling every 1 iteration
    'reg_alpha': 0.1,             # L1 regularization
    'reg_lambda': 0.1,            # L2 regularization

    # Imbalance Handling
    'is_unbalance': True,         # Automatically weights the minority (default) class

    # System
    'n_jobs': -1,                 # Uses all CPU cores
    'random_state': 42
}

for fold, (trn_idx, val_idx) in enumerate(skf.split(X,y)):
    print(trn_idx, val_idx)


[     0      1      2 ... 307507 307508 307509] [     3      6     30 ... 307492 307500 307510]
[     3      4      5 ... 307508 307509 307510] [     0      1      2 ... 307497 307501 307505]
[     0      1      2 ... 307506 307509 307510] [     5      7     10 ... 307496 307507 307508]
[     0      1      2 ... 307508 307509 307510] [     4     14     20 ... 307498 307504 307506]
[     0      1      2 ... 307507 307508 307510] [     9     11     16 ... 307502 307503 307509]
